# Phase 2 (b): SVD (Matrix Factorization)

This notebook trains a **Funk SVD** recommender on CiaoDVD. Funk SVD factorizes
the user–movie rating matrix into two low-dimensional matrices of latent
factors. A prediction for `(user, movie)` is the dot product of their factor
vectors plus user/movie biases.

We tune `n_factors` (latent dimensionality) and `reg_all` (L2 regularization)
via 3-fold cross-validation. The same `random_state=42` train/test split as in
`02_cf_model.ipynb` is used for fair comparison.


In [ ]:
import sys
from pathlib import Path

# Allow `from src.xxx import yyy` when this notebook lives in /notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


In [ ]:
import pandas as pd
from surprise import Dataset, Reader, accuracy
from surprise.model_selection import train_test_split as surprise_split

from src.svd_model import tune_svd, fit_svd

RANDOM_STATE = 42  # MUST match the KNN notebook


## 1. Load cleaned data and build the Surprise Dataset

In [ ]:
ratings = pd.read_csv("data/processed/ratings_clean.csv")

model_data = ratings[["userId", "movieId", "movieRating"]]
reader = Reader(rating_scale=(1, 5))
data   = Dataset.load_from_df(model_data, reader)

trainset, testset = surprise_split(data, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train interactions: {trainset.n_ratings:,}")
print(f"Test  interactions: {len(testset):,}")


## 2. Hyperparameter tuning (3-fold CV)

In [ ]:
best_params = tune_svd(data, cv=3)
print("Best SVD params:", best_params)


## 3. Train best SVD on the trainset and evaluate on the testset

In [ ]:
svd = fit_svd(trainset, best_params)
svd_predictions = svd.test(testset)

rmse_svd = accuracy.rmse(svd_predictions)
mae_svd  = accuracy.mae(svd_predictions)


## 4. Append metrics to results/metrics.csv

In [ ]:
import os

row = pd.DataFrame([{
    "Model": "SVD",
    "RMSE":  rmse_svd,
    "MAE":   mae_svd,
}])

path = "results/metrics.csv"
if os.path.exists(path):
    existing = pd.read_csv(path)
    existing = existing[existing["Model"] != "SVD"]
    out = pd.concat([existing, row], ignore_index=True)
else:
    out = row
out.to_csv(path, index=False)
out


**Why SVD typically wins on sparse data.** Instead of comparing raw
rating vectors (KNN), SVD learns dense, low-dimensional embeddings for users
and movies. Every rating contributes to refining the entire factor space, so
generalization is much better when the matrix is sparse. Regularization
(`reg_all`) prevents overfitting on the long-tail of users/movies with few
ratings.